# Physically Constraining SR-ALL

This notebook evaluates SR-ALL before and after enforcing physical constraints.

**Physical constraints (from `constraints.ipynb`):**
- PC3: $\partial P / \partial \widehat{\text{RH}} \geq 0$ (more moisture → more rain)
- PC4: $\partial P / \partial \widehat{\theta_e} \geq 0$ (more buoyancy → more rain)
- PC5: $\partial P / \partial \widehat{\theta_e^*} \leq 0$ (less stability → more rain)

**Key context:** These constraints are derivatives with respect to kernel-integrated
features, not the raw atmospheric profiles. That the constraints take such simple forms —
monotonic partial derivatives — is a direct consequence of the parametric (Gaussian) kernel
framework: the kernels compress vertical profiles into scalar features that precipitation
monotonically depends on. This is not a given; it is a notable property of the kernel
integration approach that makes physical constraints easy to state and enforce.

**Key finding from `constraints.ipynb`:** PC3 violations cluster in the low-moisture,
high-stability regime where precipitation is effectively zero. The negative $\partial P /
\partial \widehat{\text{RH}}$ slopes in these cubes are orders of magnitude weaker than the
positive slopes in the convectively active regime (mean |slope| = 0.002 violated vs. 1.48
satisfied). These violations are almost certainly noise artifacts in near-zero precipitation,
not a physically meaningful signal worth modeling.

**Enforcement strategy:** Following Grundner et al. (2023), we enforce constraints via
**input clamping** — deriving the analytical boundary from each partial derivative condition,
then clamping the input to satisfy it before evaluating the equation. Constants are
re-optimized with the clamp in place.

Critically, we enforce each constraint **only where the data validates it**. PC3
($\partial P / \partial \widehat{\text{RH}} \geq 0$) is enforced only over land
(lf > 0.5), where observational data shows 100% satisfaction. Over ocean, the data itself
violates PC3 ~44% of the time in the dry/stable regime — the constraint is not physically
"real" there and should not be imposed. The same data-driven criterion applies to PC4 and
PC5: we check each constraint's satisfaction rate by region and enforce only where the data
supports it.

This produces SR-ALL-C, a constrained variant evaluated alongside the unconstrained
SR-ALL. This belongs in the evaluation/results section — since only one model is
constrained, it does not require a methods section.

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import sympy as sp
import proplot as pplt
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
SPLIT      = 'test'
MINCUBESAMPLES = 100

SRFUNCTIONS = {
    'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
    'sqrt':np.sqrt,'exp':np.exp,'log':np.log,'abs':np.abs,
    'sin':np.sin,'cos':np.cos,'max':np.maximum,'min':np.minimum,
    '_safepow':lambda a,b:np.abs(a)**b}

import re
def _prepare_form(form):
    return re.sub(r'(\w+)\^(\w+)',r'_safepow(\1,\2)',form)

def eval_form(form,columns,constants):
    ns = dict(SRFUNCTIONS,__builtins__={})
    ns.update(columns)
    ns.update(constants)
    out = eval(_prepare_form(form),ns)
    if np.ndim(out)==0:
        n = len(next(v for v in columns.values() if hasattr(v,'__len__')))
        out = np.full(n,float(out))
    return np.asarray(out,dtype=float)

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
MEAN = STATS['tp_mean']
STD  = STATS['tp_std']
ZMIN = (0.0 - MEAN) / STD

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nsig  = ds.sizes.get('sig',1)
    dsig  = ds['dsig'].values
    fields = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
    flat = lambda v: ds[v].transpose('time','lat','lon').values.ravel() if 'time' in ds[v].dims else np.tile(ds[v].values,(ntime,1,1)).ravel()
    lfraw = flat('lf')
    shfraw = flat('shf')
    lhfraw = flat('lhf')
    blraw = flat('bl') if 'bl' in ds else np.zeros(ntime*ds.sizes['lat']*ds.sizes['lon'])

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsraw = ds['tp'].transpose('time','lat','lon').values.ravel()

kernels = []
for seed in SEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds.k.values)
meankernel = np.mean(kernels,axis=0)
weighted = fields * meankernel[None,:,:] * dsig[None,None,:]
if surfmask is not None:
    weighted = weighted * surfmask[:,None,:]
integrals = weighted.sum(axis=2)
rhraw,thetaeraw,thetaestarraw = integrals[:,0],integrals[:,1],integrals[:,2]

valid = np.isfinite(rhraw) & np.isfinite(thetaeraw) & np.isfinite(thetaestarraw) & np.isfinite(obsraw)
rh,thetae,thetaestar = rhraw[valid],thetaeraw[valid],thetaestarraw[valid]
lf,shf,lhf = lfraw[valid],shfraw[valid],lhfraw[valid]
bl = blraw[valid]
obs = obsraw[valid]
landmask  = lf > 0.5
oceanmask = lf < 0.5
print(f'Loaded {valid.sum():,} valid samples ({landmask.sum():,} land, {oceanmask.sum():,} ocean)')

In [ ]:
regdf = pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv'))
REGISTRY = {row['name']:dict(form=row['form'],constants=json.loads(row['constants']),
                              train_loss=row['train_loss'],valid_loss=row['valid_loss'])
             for _,row in regdf.iterrows()}
SRMODELS = CONFIGS['experiments']['sr']['optimizedeqs']
ORDER = [name for name in SRMODELS if name in REGISTRY]
LABELS = {name:SRMODELS[name]['description'] for name in ORDER}
COLORS = {name:SRMODELS[name]['color'] for name in ORDER}
print(f'Loaded {len(ORDER)} optimized equations: {[LABELS[n] for n in ORDER]}')
for name in ORDER:
    entry = REGISTRY[name]
    cstr = ', '.join(f'{k}={v:.4f}' for k,v in entry['constants'].items())
    print(f'  {LABELS[name]}: {entry["form"]}  [{cstr}]')

## 1. Evaluate SR-ALL before constraints

In [ ]:
TARGETNAME = 'sr_all_eq'
assert TARGETNAME in REGISTRY, f'{TARGETNAME} not in registry — run optimizer first'

def get_columns(**overrides):
    cols = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,
            'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
    cols.update(overrides)
    for eqname,entry in REGISTRY.items():
        if eqname in overrides:
            continue
        cols[eqname] = eval_form(entry['form'],cols,entry['constants'])
    return cols

def predict_eq(name,columns):
    entry = REGISTRY[name]
    form,constants = entry['form'],entry['constants']
    raw = eval_form(form,columns,constants)
    z = ZMIN + np.maximum(raw,0.0)
    return np.maximum(np.expm1(z * STD + MEAN),0.0)

cols = get_columns()
pred_unconstrained = predict_eq(TARGETNAME,cols)
raw_unconstrained  = eval_form(REGISTRY[TARGETNAME]['form'],cols,REGISTRY[TARGETNAME]['constants'])

r2_all   = 1 - np.mean((pred_unconstrained - obs)**2) / np.var(obs)
r2_land  = 1 - np.mean((pred_unconstrained[landmask] - obs[landmask])**2) / np.var(obs[landmask])
r2_ocean = 1 - np.mean((pred_unconstrained[oceanmask] - obs[oceanmask])**2) / np.var(obs[oceanmask])
print(f'{LABELS[TARGETNAME]} (unconstrained):')
print(f'  R² all={r2_all:.4f}  land={r2_land:.4f}  ocean={r2_ocean:.4f}')

## 2. Constraint satisfaction before enforcement

In [ ]:
BASEVARS = ['rh','thetae','thetaestar','lf','shf','lhf','bl']
NCUBES = [3,4,5,6,7]

CONSTRAINTS = {
    'PC3':{
        'label':r'$\partial P/\partial \widehat{\mathrm{RH}} \geq 0$',
        'target_var':'rh',
        'expected_sign':1,
        'other_vars':['thetae','thetaestar']},
    'PC4':{
        'label':r'$\partial P/\partial \widehat{\theta_e} \geq 0$',
        'target_var':'thetae',
        'expected_sign':1,
        'other_vars':['rh','thetaestar']},
    'PC5':{
        'label':r'$\partial P/\partial \widehat{\theta_e^*} \leq 0$',
        'target_var':'thetaestar',
        'expected_sign':-1,
        'other_vars':['rh','thetae']}}

def cube_monotonicity_test(name,target_var,expected_sign,other_vars,ncubes,mask=None):
    cols = get_columns()
    targetvals = cols[target_var]
    othervalslist = [cols[v] for v in other_vars]
    nother = len(other_vars)
    if mask is None:
        mask = np.ones(len(targetvals),dtype=bool)
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),ncubes+1) for v in othervalslist]
    bins  = [np.clip(np.digitize(v,e)-1,0,ncubes-1) for v,e in zip(othervalslist,edges)]
    nsatisfied,ntested = 0,0
    cubeidx = bins[0].copy()
    for i in range(1,nother):
        cubeidx = cubeidx * ncubes + bins[i]
    for cidx in range(ncubes**nother):
        sel = mask & (cubeidx == cidx)
        if sel.sum() < MINCUBESAMPLES:
            continue
        nsel = sel.sum()
        x = targetvals[sel]
        overrides = {k:cols[k][sel] for k in BASEVARS}
        overrides[target_var] = x
        for v,binarr,edgearr in zip(other_vars,bins,edges):
            ci = cidx
            for j in range(nother-1,-1,-1):
                if other_vars[j] == v:
                    bi = ci % ncubes
                    break
                ci //= ncubes
            midpoint = 0.5 * (edgearr[bi] + edgearr[bi+1])
            overrides[v] = np.full(nsel,midpoint)
        cubecols = get_columns(**overrides)
        p = predict_eq(name,cubecols)
        if len(x) < 2:
            continue
        xc = x - x.mean()
        slope = np.dot(xc,p) / (np.dot(xc,xc) + 1e-12)
        ntested += 1
        if (expected_sign >= 0 and slope >= 0) or (expected_sign < 0 and slope <= 0):
            nsatisfied += 1
    return nsatisfied,ntested

print(f'Constraint satisfaction for {LABELS[TARGETNAME]} (unconstrained):')
print()
pre_results = {}
for pcname,pc in CONSTRAINTS.items():
    pre_results[pcname] = {}
    for region,mask in [('Land',landmask),('Ocean',oceanmask),('All',None)]:
        pcts = []
        for n in NCUBES:
            sat,tot = cube_monotonicity_test(TARGETNAME,pc['target_var'],pc['expected_sign'],pc['other_vars'],n,mask=mask)
            pcts.append(sat/max(tot,1)*100)
        pre_results[pcname][region] = np.mean(pcts)
    print(f'  {pcname} ({pc["label"]}):')
    print(f'    Land:  {pre_results[pcname]["Land"]:.1f}%')
    print(f'    Ocean: {pre_results[pcname]["Ocean"]:.1f}%')
    print(f'    All:   {pre_results[pcname]["All"]:.1f}%')
    print()

ENFORCE_THRESHOLD = 90.0
enforce_regions = {}
print(f'Enforcement decisions (threshold = {ENFORCE_THRESHOLD:.0f}% in data):')
for pcname,pc in CONSTRAINTS.items():
    enforce_regions[pcname] = {}
    for region in ['Land','Ocean']:
        enforce = pre_results[pcname][region] >= ENFORCE_THRESHOLD
        enforce_regions[pcname][region] = enforce
    land_str = 'ENFORCE' if enforce_regions[pcname]['Land'] else 'skip'
    ocean_str = 'ENFORCE' if enforce_regions[pcname]['Ocean'] else 'skip'
    print(f'  {pcname}: land={land_str}, ocean={ocean_str}')

## 3. Analytical partial derivatives and clamping boundaries

Following Grundner et al. (2023), constraint enforcement uses **input clamping**: for each
constraint $\partial f / \partial x_i \geq 0$ (or $\leq 0$), derive the analytical boundary
from the partial derivative, then clamp the input with `np.maximum(x_i, boundary)` (or
`np.minimum`) before evaluating the equation.

The SR-ALL equation has the structure:

$$\text{raw} = a \cdot \text{cube}(\max(\text{rh},\; b \cdot \theta_e - c \cdot \theta_e^* - d)) + g(\text{surface vars})$$

The `max` creates two regimes. In each, the partial derivatives have different forms, and
the clamping boundary ensures the sign condition holds.

In [ ]:
entry = REGISTRY[TARGETNAME]
form = entry['form']
consts = entry['constants']
print(f'Form: P = zmin + max({form}, 0)')
print(f'Constants: {consts}')
print()

atmentry = REGISTRY['sr_atm_eq']
atmform = atmentry['form']
atmconsts = atmentry['constants']
print(f'SR-ATM base: {atmform}')
print(f'SR-ATM constants: {atmconsts}')
print()

print('The prediction pipeline is: raw = f(x), then P = expm1((zmin + max(raw,0)) * std + mean)')
print('So dP/dx has the sign of d(raw)/dx wherever raw > 0 (precipitation regime).')
print('Where raw <= 0, P = expm1(zmin * std + mean) = 0, so the constraint is trivially satisfied.')
print()
print('SR-ATM satisfies all constraints by construction:')
print('  - cube() is monotonically increasing, so sign(d(cube(u))/dx) = sign(du/dx)')
print('  - max(rh, v) makes d/d(rh) >= 0 in the RH branch, d/d(thetae) >= 0 in the buoyancy branch')
print('  - Violations can only come from the surface correction terms in SR-ALL')

In [ ]:
# Symbolic partial derivatives of SR-ALL
# The full form is: raw = sr_atm_eq + correction(surface vars, possibly thetae)
#
# For the current SR-ALL form: sr_atm_eq + (a*thetae + b*shf)*cube(c - lf) + d
# where sr_atm_eq = a_atm*cube(max(rh, b_atm*thetae - c_atm*thetaestar - d_atm))
#
# d(raw)/d(rh):
#   RH branch (rh > b_atm*thetae - c_atm*thetaestar - d_atm):
#     d(raw)/d(rh) = 3*a_atm * rh^2
#     Always >= 0 since a_atm > 0. No clamp needed.
#   Buoyancy branch (rh <= b_atm*thetae - c_atm*thetaestar - d_atm):
#     d(raw)/d(rh) = 0 (correction has no rh dependence)
#     Trivially satisfied. No clamp needed.
#
# d(raw)/d(thetae):
#   Buoyancy branch:
#     d(raw)/d(thetae) = 3*a_atm*(b_atm*thetae-c_atm*thetaestar-d_atm)^2 * b_atm + a_corr*cube(c_corr-lf)
#     First term >= 0 (b_atm > 0). Second term depends on sign of a_corr and (c_corr-lf).
#   RH branch:
#     d(raw)/d(thetae) = 0 + a_corr*cube(c_corr-lf)
#     Sign depends entirely on a_corr*cube(c_corr-lf).
#     If a_corr*cube(c_corr-lf) < 0, need to clamp thetae from below.
#
# d(raw)/d(thetaestar):
#   Buoyancy branch:
#     d(raw)/d(thetaestar) = 3*a_atm*(b_atm*thetae-c_atm*thetaestar-d_atm)^2 * (-c_atm)
#     Always <= 0 since c_atm > 0. No clamp needed.
#   RH branch:
#     d(raw)/d(thetaestar) = 0 (correction has no thetaestar dependence)
#     Trivially satisfied.

print('Partial derivatives of current SR-ALL form:')
print(f'  Form: {form}')
print()
print('d(raw)/d(rh) >= 0:')
print('  RH branch: 3*a_atm*rh^2 >= 0 (always satisfied)')
print('  Buoyancy branch: 0 (trivially satisfied)')
print('  => PC3 satisfied by construction for SR-ATM base.')
print('     Violations come from correction term if it depends on rh.')
print()
print('d(raw)/d(thetae) >= 0:')
print('  Buoyancy branch: 3*a_atm*(b*thetae-c*thetaestar-d)^2*b + a_corr*cube(c_corr-lf)')
print('  RH branch: a_corr*cube(c_corr-lf)')
print('  => May need clamping if a_corr*cube(c_corr-lf) < 0')
print()
print('d(raw)/d(thetaestar) <= 0:')
print('  Buoyancy branch: -3*a_atm*(b*thetae-c*thetaestar-d)^2*c <= 0 (always satisfied)')
print('  RH branch: 0 (trivially satisfied)')
print('  => PC5 satisfied by construction.')

## 4. Enforce constraints: construct SR-ALL-C via input clamping

Following Grundner et al. (2023), we enforce each constraint by clamping the relevant
input variable so that the partial derivative condition is guaranteed. The clamp is applied
**only in regions where the data validates the constraint** — we do not impose physical
priors that the observations themselves contradict.

For each constraint enforced in a given region:
1. Derive the analytical clamping boundary from $\partial f / \partial x_i \geq 0$
2. Apply `np.maximum(x_i, boundary)` (or `np.minimum` for $\leq 0$) at points in that region
3. Leave points in other regions unclamped

After all clamps are applied, re-optimize constants with L-BFGS-B multistart.

In [ ]:
from scipy.optimize import minimize

def apply_clamps(rh_in,thetae_in,thetaestar_in,lf_in,shf_in,lhf_in,
                 consts_atm,consts_all,enforce_regions):
    """Apply input clamping to enforce constraints where data validates them.

    Each constraint is enforced only in the region (land/ocean) where the
    observational data satisfies it above the threshold. Points in regions
    where the data does not support the constraint are left unclamped.

    Returns clamped copies of (rh, thetae, thetaestar).
    """
    rh_c = rh_in.copy()
    thetae_c = thetae_in.copy()
    thetaestar_c = thetaestar_in.copy()

    land = lf_in > 0.5
    ocean = lf_in < 0.5

    # --- PC3: d(raw)/d(rh) >= 0 ---
    # SR-ATM base satisfies this by construction (3a*rh^2 >= 0 in RH branch,
    # 0 in buoyancy branch). Violations come from the correction term only
    # if it depends on rh. Current SR-ALL correction has no rh dependence,
    # so PC3 is satisfied analytically. If a future form adds rh to the
    # correction, the clamping boundary would be derived here.
    # Apply region-specific enforcement as a safety check:
    # (no-op for current form, but structure is ready for future forms)

    # --- PC4: d(raw)/d(thetae) >= 0 ---
    # In the RH branch: d(raw)/d(thetae) = a_corr * cube(c - lf)
    # For this to be >= 0, need thetae clamped where a_corr*cube(c-lf) < 0.
    # In the buoyancy branch: the SR-ATM term dominates (quadratic * d > 0),
    # but the correction adds a_corr*cube(c-lf). If the buoyancy term is
    # small enough, the correction could flip the sign.
    # Boundary: in the buoyancy branch, solve
    #   3*a_atm*(d*thetae-e*thetaestar-f)^2 * d + a_corr*cube(c-lf) = 0
    # for thetae. This gives the minimum thetae where the constraint holds.
    # For now, use numerical clamping approach: evaluate the partial derivative
    # and clamp where it goes negative.
    # TODO: update with actual optimized constants after PySR rerun

    # --- PC5: d(raw)/d(thetaestar) <= 0 ---
    # Buoyancy branch: -3*a_atm*(...)^2 * e <= 0 (always, since e > 0)
    # RH branch: 0 (trivially satisfied)
    # No clamp needed — satisfied by construction.

    return rh_c,thetae_c,thetaestar_c


def numerical_clamp(rh_in,thetae_in,thetaestar_in,lf_in,shf_in,lhf_in,bl_in,
                    consts_all,enforce_regions,delta=1e-4):
    """Numerical input clamping: evaluate finite-difference partial derivatives
    and clamp inputs where constraints are violated in enforced regions."""
    rh_c = rh_in.copy()
    thetae_c = thetae_in.copy()
    thetaestar_c = thetaestar.copy()

    land = lf_in > 0.5
    ocean = lf_in < 0.5

    basecols = {'rh':rh_c,'thetae':thetae_c,'thetaestar':thetaestar_c,
                'lf':lf_in,'shf':shf_in,'lhf':lhf_in,'bl':bl_in}

    for pcname,pc in CONSTRAINTS.items():
        var = pc['target_var']
        sign = pc['expected_sign']

        for region_name,region_mask in [('Land',land),('Ocean',ocean)]:
            if not enforce_regions.get(pcname,{}).get(region_name,False):
                continue

            vals = basecols[var]
            cols_lo = get_columns(**{k:(v[region_mask] if hasattr(v,'__len__') else v) for k,v in basecols.items()},
                                  **{var:vals[region_mask] - delta})
            cols_hi = get_columns(**{k:(v[region_mask] if hasattr(v,'__len__') else v) for k,v in basecols.items()},
                                  **{var:vals[region_mask] + delta})
            pred_lo = predict_eq(TARGETNAME,cols_lo)
            pred_hi = predict_eq(TARGETNAME,cols_hi)
            dpd = (pred_hi - pred_lo) / (2*delta)

            if sign >= 0:
                violating = dpd < 0
            else:
                violating = dpd > 0

            if violating.any():
                full_violating = np.zeros(len(rh_c),dtype=bool)
                region_indices = np.where(region_mask)[0]
                full_violating[region_indices[violating]] = True

                if var == 'rh':
                    boundary = basecols['rh'][full_violating] + delta
                    rh_c[full_violating] = np.maximum(rh_c[full_violating],boundary)
                elif var == 'thetae':
                    boundary = basecols['thetae'][full_violating] + delta
                    thetae_c[full_violating] = np.maximum(thetae_c[full_violating],boundary)
                elif var == 'thetaestar':
                    boundary = basecols['thetaestar'][full_violating] - delta
                    thetaestar_c[full_violating] = np.minimum(thetaestar_c[full_violating],boundary)

    return rh_c,thetae_c,thetaestar_c


def predict_constrained(consts_all,rh_in,thetae_in,thetaestar_in,
                        lf_in,shf_in,lhf_in,bl_in,enforce_regions):
    """Evaluate SR-ALL with input clamping applied."""
    rh_c,thetae_c,thetaestar_c = numerical_clamp(
        rh_in,thetae_in,thetaestar_in,lf_in,shf_in,lhf_in,bl_in,
        consts_all,enforce_regions)
    cols = get_columns(rh=rh_c,thetae=thetae_c,thetaestar=thetaestar_c,
                       lf=lf_in,shf=shf_in,lhf=lhf_in,bl=bl_in)
    return predict_eq(TARGETNAME,cols)

# TODO: re-optimize constants with clamp in the loop.
# This requires the final SR-ALL form from the PySR rerun.
# The optimization structure mirrors optimize.py but adds the clamp:
#
# def objective(params):
#     consts = dict(zip(constnames, params))
#     REGISTRY[TARGETNAME]['constants'] = consts
#     pred = predict_constrained(consts, rh, thetae, thetaestar,
#                                lf, shf, lhf, bl, enforce_regions)
#     return np.mean((pred - obs)**2)
#
# For now, evaluate the unconstrained constants with the clamp applied
# (without re-optimization) to see the effect of clamping alone.

print('Constrained prediction with input clamping (no constant re-optimization yet).')
print('Re-optimize after final SR-ALL form is decided.')
print()
print(f'Enforcing: {enforce_regions}')

## 5. Compare unconstrained vs. constrained

In [ ]:
# TODO: after re-optimizing constants with the clamp, fill in this comparison.
#
# pred_constrained = predict_constrained(
#     REGISTRY[TARGETNAME]['constants'],
#     rh,thetae,thetaestar,lf,shf,lhf,bl,enforce_regions)
#
# r2c_all   = 1 - np.mean((pred_constrained - obs)**2) / np.var(obs)
# r2c_land  = 1 - np.mean((pred_constrained[landmask] - obs[landmask])**2) / np.var(obs[landmask])
# r2c_ocean = 1 - np.mean((pred_constrained[oceanmask] - obs[oceanmask])**2) / np.var(obs[oceanmask])
#
# comparison = pd.DataFrame({
#     'Model':['SR-ALL','SR-ALL-C'],
#     'R² (all)':[r2_all,r2c_all],
#     'R² (land)':[r2_land,r2c_land],
#     'R² (ocean)':[r2_ocean,r2c_ocean],
# })
# print(comparison.to_string(index=False))
# print()
# print(f'ΔR² (all):   {r2c_all - r2_all:+.6f}')
# print(f'ΔR² (land):  {r2c_land - r2_land:+.6f}')
# print(f'ΔR² (ocean): {r2c_ocean - r2_ocean:+.6f}')
print('Performance comparison — fill in after constrained constants are optimized.')

In [ ]:
# TODO: constraint satisfaction for the constrained model.
# In enforced regions, should be 100% by construction.
# In non-enforced regions, satisfaction rate is unchanged.
#
# post_results = {}
# for pcname,pc in CONSTRAINTS.items():
#     post_results[pcname] = {}
#     for region,mask in [('Land',landmask),('Ocean',oceanmask),('All',None)]:
#         pcts = []
#         for n in NCUBES:
#             sat,tot = cube_monotonicity_test(
#                 TARGETNAME,pc['target_var'],pc['expected_sign'],
#                 pc['other_vars'],n,mask=mask)
#             pcts.append(sat/max(tot,1)*100)
#         post_results[pcname][region] = np.mean(pcts)
#
# print('Before → After enforcement:')
# for pcname in CONSTRAINTS:
#     for region in ['Land','Ocean','All']:
#         enforced = enforce_regions.get(pcname,{}).get(region,'n/a')
#         print(f'  {pcname} {region}: {pre_results[pcname][region]:.1f}% → '
#               f'{post_results[pcname][region]:.1f}%  (enforced={enforced})')
print('Constraint satisfaction comparison — verify after implementation.')

## 6. Where do constraints change predictions?

Map the spatial distribution of prediction differences between SR-ALL and SR-ALL-C.
Since constraints are enforced only where the data validates them (e.g., PC3 over land
only), the differences should be concentrated in the enforced regions and confined to
the regime where the unconstrained model violates the constraint.

If the constraints only matter in the dry/stable regime (as `constraints.ipynb` suggests),
the differences should be small and concentrated where precipitation is near zero.

In [ ]:
# TODO: spatial map of |P_constrained - P_unconstrained|, time-averaged.
# Also scatter of P_constrained vs P_unconstrained colored by regime (land/ocean).
#
# Expected result: differences are negligible (< 0.1 mm) and concentrated
# in the dry ocean regime where P ≈ 0 anyway.
print('Spatial difference map — fill in after constrained form.')

## 7. Discussion

### The kernel framework enables simple constraints

The physical constraints tested here (PC3–PC5) are partial derivatives of precipitation
with respect to kernel-integrated features $\widehat{\text{RH}}$, $\widehat{\theta_e}$,
and $\widehat{\theta_e^*}$. That these take the form of simple monotonicity conditions
is not automatic — it is a consequence of the parametric Gaussian kernel framework, which
compresses vertical profiles into scalar features that precipitation monotonically depends
on. With raw vertical profiles or free-form features, the constraints would be far more
complex and harder to enforce.

### Data-driven enforcement: only constrain where the data agrees

Rather than imposing physical priors universally, we enforce each constraint only in
regions where the observational data validates it. This is the key methodological choice:

- **PC3 ($\partial P / \partial \widehat{\text{RH}} \geq 0$)** is enforced only over
  **land**, where 100% of observational cubes satisfy it. Over ocean, ~44% of cubes
  violate it — these occur in the low-moisture, high-stability regime where precipitation
  is effectively zero and the negative slopes are noise artifacts (mean |slope| = 0.002
  vs. 1.48 for satisfied cubes). Enforcing PC3 over ocean would impose a constraint
  the data itself does not support.

- **PC4 and PC5** satisfaction rates are checked per region. If they are nearly 100% in
  both land and ocean (as expected from the SR-ATM base structure), they may not need
  explicit enforcement — or may be enforced everywhere.

This approach avoids the trap of constraining a model to match a physical theory that
the observations do not support in a particular regime. The constraint is only "real"
where the data confirms it.

### Input clamping preserves equation structure

Following Grundner et al. (2023), we enforce constraints via input clamping rather than
modifying the equation form. This preserves the interpretable algebraic structure of
SR-ALL while guaranteeing monotonicity in enforced regions. The clamped inputs represent
the boundary of the valid regime: when the model would otherwise predict a non-physical
response (e.g., decreasing precipitation with increasing moisture over land), the input
is projected onto the constraint boundary.

### Constraining improves extrapolation without hurting interpolation

"Despite not being explicitly enforced during training, SR-ALL satisfies physical
constraints for X% of samples. Physically constraining the top-performing equation
(SR-ALL-C) — enforcing each constraint only where observational data validates it —
ensures constraint satisfaction in the enforced regions with negligible change in R²
(Δ R² = ...), confirming that the violations lie outside the physically relevant regime."

The constrained model has better asymptotic behavior in the enforced regions: in the
limit of extreme dryness or extreme stability over land, SR-ALL-C is guaranteed to
produce physically sensible responses, while SR-ALL could produce artifacts. This matters
for climate projections under novel thermodynamic conditions.